In [16]:
import math
import torch
from torch import nn, optim

In [17]:
#fourier embedding
class Fourier_embedding(nn.Module):
    #initialize
    def __init__(self, embed_dim=256, scale=16):
        super().__init__()
        #register buffer
        self.register_buffer(
            'freqs',
            torch.randn(embed_dim // 2) * scale
        )
    #forward propagation
    def forward(self, sigma):
        #change the shape of sigma
        if sigma.dim() > 1:
            sigma = sigma.view(-1)
        #log -> sigma
        c_noise = torch.log(sigma) / 4
        args = c_noise[:, None] * self.freqs[None, :] * 2 * math.pi
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return emb

In [18]:
#time MLP
class Time_MLP(nn.Module):
    #initialize
    def __init__(self, fourier_dim=256, time_dim=512):
        super().__init__()
        #MLP layer
        self.net = nn.Sequential(
            nn.Linear(in_features=fourier_dim, out_features=time_dim),
            nn.SiLU(),
            nn.Linear(in_features=time_dim, out_features=time_dim)
        )
    #forward propagation
    def forward(self, emb):
        return self.net(emb)

In [19]:
#Residual Block
class residual_block(nn.Module):
    #initialize
    def __init__(self, in_channels, out_channels, time_dim, stride):
        super().__init__()
        #layer 1
        self.conv_1 = nn.Sequential(
            nn.GroupNorm(num_groups=32, num_channels=in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1),
        )
        #layer 2
        self.norm_2 = nn.GroupNorm(num_groups=32, num_channels=out_channels)
        self.act_2 = nn.SiLU()
        self.conv_2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1)
        #time projection layer
        self.time_proj = nn.Linear(time_dim, out_channels)
        #choose skip way
        if in_channels != out_channels or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1, stride=stride, bias=False)
            )
        else:
            self.skip = nn.Identity()
    #forward propagation
    def forward(self, x, t):
        #layer 1
        h = self.conv_1(x)
        #time projection layer
        time_emb = self.time_proj(t)
        time_emb = time_emb[:, :, None, None]
        h = h + time_emb
        #layer 2
        h = self.norm_2(h)
        h = self.act_2(h)
        h = self.conv_2(h)
        return h + self.skip(x)

In [20]:
#Downsample/Upsample
class downsample(nn.Module):
    #initialize
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=2, padding=1)
    #forward propagation
    def forward(self, x):
        return self.conv(x)

class upsample(nn.Module):
    #initialize
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1)
    #forward propagation
    def forward(self, x):
        x = nn.functional.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)

In [21]:
#attention block
class Attention_block(nn.Module):
    #initialize
    def __init__(self, channels):
        super().__init__()
        #layer norm
        self.norm = nn.GroupNorm(num_groups=32, num_channels=channels)
        #layer query, key and value
        self.q = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=1)
        self.k = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=1)
        self.v = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=1)
        #layer projection
        self.proj = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=1)
    #forward propagation
    def forward(self, x):
        #get dimensional info
        B, C, H, W = x.shape
        #layer norm
        h = self.norm(x)
        #layer query, key and value
        q = self.q(h).reshape(B, C, H*W)
        k = self.k(h).reshape(B, C, H*W)
        v = self.v(h).reshape(B, C, H*W)
        #change shape of query
        q = q.permute(0, 2, 1)
        #calculate attention value
        attn = torch.bmm(q, k)
        attn = attn * (C ** -0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.bmm(attn, v.permute(0, 2, 1))
        out = out.permute(0, 2, 1).reshape(B, C, H, W)
        #layer projection
        out = self.proj(out)
        return x + out

In [22]:
#UNet
class UNet(nn.Module):
    #initialize
    def __init__(self, base_channel=128, time_dim=512):
        super().__init__()
        #initialize relative variables
        self.time_dim = time_dim
        self.fourier = Fourier_embedding()
        self.time_mlp = Time_MLP()
        #init layer
        self.init_conv = nn.Conv2d(in_channels=3, out_channels=base_channel, kernel_size=3, padding=1)
        #down stage 1
        self.res_1_1 = residual_block(in_channels=base_channel, out_channels=base_channel, time_dim=self.time_dim, stride=1)
        self.res_1_2 = residual_block(in_channels=base_channel, out_channels=base_channel, time_dim=self.time_dim, stride=1)
        self.down_1 = downsample(channels=base_channel)
        #down stage 2
        self.res_2_1 = residual_block(in_channels=base_channel, out_channels=base_channel * 2, time_dim=self.time_dim, stride=1)
        self.res_2_2 = residual_block(in_channels=base_channel * 2, out_channels=base_channel * 2, time_dim=self.time_dim, stride=1)
        self.attn_2 = Attention_block(channels=base_channel * 2)
        self.down_2 = downsample(channels=base_channel * 2)
        #down stage 3
        self.res_3_1 = residual_block(in_channels=base_channel * 2, out_channels=base_channel * 4, time_dim=self.time_dim, stride=1)
        self.res_3_2 = residual_block(in_channels=base_channel * 4, out_channels=base_channel * 4, time_dim=self.time_dim, stride=1)
        self.attn_3 = Attention_block(channels=base_channel * 4)
        self.down_3 = downsample(channels=base_channel * 4)
        #mid layer
        self.mid_1 = residual_block(in_channels=base_channel * 4, out_channels=base_channel * 4, time_dim=self.time_dim, stride=1)
        self.mid_2 = residual_block(in_channels=base_channel * 4, out_channels=base_channel * 4, time_dim=self.time_dim, stride=1)
        #up stage 3
        self.up_3 = upsample(channels=base_channel * 4)
        self.res_4_1 = residual_block(in_channels=base_channel * 4 + base_channel * 4, out_channels=base_channel * 4, time_dim=self.time_dim, stride=1)
        self.res_4_2 = residual_block(in_channels=base_channel * 4, out_channels=base_channel * 4, time_dim=self.time_dim, stride=1)
        #up stage 2
        self.up_2 = upsample(channels=base_channel * 4)
        self.res_5_1 = residual_block(in_channels=base_channel * 4 + base_channel * 2, out_channels=base_channel * 2, time_dim=self.time_dim, stride=1)
        self.res_5_2 = residual_block(in_channels=base_channel * 2, out_channels=base_channel * 2, time_dim=self.time_dim, stride=1)
        #up stage 1
        self.up_1 = upsample(channels=base_channel * 2)
        self.res_6_1 = residual_block(in_channels=base_channel * 2 + base_channel, out_channels=base_channel, time_dim=self.time_dim, stride=1)
        self.res_6_2 = residual_block(in_channels=base_channel, out_channels=base_channel, time_dim=self.time_dim, stride=1)
        #final layer
        self.final_conv = nn.Conv2d(in_channels=base_channel, out_channels=3, kernel_size=3, padding=1)
    #forward propagation
    def forward(self, x, sigma):
        #time embedding
        t = self.fourier(sigma)
        t = self.time_mlp(t)
        #init layer
        x = self.init_conv(x)
        #down stage 1
        h_1 = self.res_1_1(x, t)
        h_1 = self.res_1_2(h_1, t)
        d_1 = self.down_1(h_1)
        #down stage 2
        h_2 = self.res_2_1(d_1, t)
        h_2 = self.res_2_2(h_2, t)
        h_2 = self.attn_2(h_2)
        d_2 = self.down_2(h_2)
        #down stage 3
        h_3 = self.res_3_1(d_2, t)
        h_3 = self.res_3_2(h_3, t)
        h_3 = self.attn_3(h_3)
        d_3 = self.down_3(h_3)
        #mid layer
        mid = self.mid_1(d_3, t)
        mid = self.mid_2(mid, t)
        #up stage 3
        u_3 = self.up_3(mid)
        u_3 = self.res_4_1(torch.cat([u_3, h_3], dim=1), t)
        u_3 = self.res_4_2(u_3, t)
        #up stage 2
        u_2 = self.up_2(u_3)
        u_2 = self.res_5_1(torch.cat([u_2, h_2], dim=1), t)
        u_2 = self.res_5_2(u_2, t)
        #up stage 1
        u_1 = self.up_1(u_2)
        u_1 = self.res_6_1(torch.cat([u_1, h_1], dim=1), t)
        u_1 = self.res_6_2(u_1, t)
        #final layer
        out = self.final_conv(u_1)
        return out